## 🔹 Part 1 — Data Processing

In [ ]:
# Load the datasets
import pandas as pd
import os

# Define base folder path
DATA_PATH = "store_sales_data"

# Load datasets using os.path.join
train_df = pd.read_csv(os.path.join(DATA_PATH, "train.csv"), parse_dates=["date"])
test_df = pd.read_csv(os.path.join(DATA_PATH, "test.csv"), parse_dates=["date"])
store_df = pd.read_csv(os.path.join(DATA_PATH, "stores.csv"))
oil_df = pd.read_csv(os.path.join(DATA_PATH, "oil.csv"), parse_dates=["date"])
transactions_df = pd.read_csv(os.path.join(DATA_PATH, "transactions.csv"), parse_dates=["date"])
holidays_events_df = pd.read_csv(os.path.join(DATA_PATH, "holidays_events.csv"), parse_dates=["date"])


### Check top five rows of each dataframe

In [ ]:

train_df.head(5)

In [ ]:
test_df.head(5)

In [ ]:
transactions_df.head(5)

In [ ]:
store_df.head(5)

In [ ]:
oil_df.head()

In [ ]:
holidays_events_df.head(5)

In [ ]:
# Create a dictionary of your DataFrames
all_dfs = {
    "train": train_df,
    "test": test_df,
    "store": store_df,
    "transactions": transactions_df,
    "oil": oil_df,
    "holidays_events": holidays_events_df
}

# Loop through the dictionary and print the null counts
print("Total Missing Values per DataFrame:")
for name, df in all_dfs.items():
    total_nulls = df.isnull().sum().sum()
    print(f"- {name}_df: {total_nulls}")

In [ ]:
oil_df['dcoilwtico'] = oil_df['dcoilwtico'].interpolate()
oil_df['dcoilwtico'] = oil_df['dcoilwtico'].fillna(method='bfill') # type: ignore

In [ ]:
oil_df.isnull().sum()

In [ ]:
train_df.columns

In [ ]:
train_df.describe()

In [ ]:
len(train_df["family"].unique())

In [ ]:
test_df.describe()

In [ ]:
print(train_df.shape)
print(oil_df.shape)

In [ ]:
train_oil_df = train_df.merge(oil_df, on="date", how="left")

In [ ]:
train_oil_df.isnull().sum().sum()

In [ ]:
train_oil_df.isnull().sum()

In [ ]:
train_oil_df["dcoilwtico"] = train_oil_df["dcoilwtico"].fillna(method="bfill") # type: ignore

In [ ]:
train_oil_df.isnull().sum()

In [ ]:
print(f"train_df shape: {train_df.shape}")
print(f"train_oil_df shape: {train_oil_df.shape}")

In [ ]:
holidays_events_df.head(5)

In [ ]:
holidays_events_df["type"].unique()

In [ ]:
holidays_events_df.shape

In [ ]:
store_df.columns

In [ ]:
train_oil_store_df = train_oil_df.merge(store_df, on='store_nbr', how='left')
train_oil_store_df.head(5)

In [ ]:
train_oil_store_df.isnull().sum().sum()

In [ ]:
holidays_events_df.info()

In [ ]:
# removed tranfereed columns
holidays_events_df = holidays_events_df[holidays_events_df['transferred'] == False]
holidays_events_df.shape

In [ ]:
holidays_events_df['type'].unique()

In [ ]:
holidays_events_df = holidays_events_df[holidays_events_df['type'] != 'Work Day']
holidays_events_df.shape

In [ ]:
train_oil_store_df.head(5)

In [ ]:
holidays_events_df.columns

In [ ]:
holidays_events_df['locale'].unique()

In [ ]:
train_oil_store_holiday_df = train_oil_store_df.merge(holidays_events_df, on='date', how='left')

In [ ]:
train_oil_store_holiday_df.head(5)

In [ ]:
train_oil_store_holiday_df['holiday_flag'] = 0

In [ ]:
train_oil_store_holiday_df.loc[train_oil_store_holiday_df['locale'] == 'National', 'holiday_flag'] = 1

train_oil_store_holiday_df.loc[
    (train_oil_store_holiday_df['locale'] == 'Regional') &
    (train_oil_store_holiday_df['locale_name'] == train_oil_store_holiday_df['state']),
    'holiday_flag'
] = 1

train_oil_store_holiday_df.loc[
    (train_oil_store_holiday_df['locale'] == 'Local') &
    (train_oil_store_holiday_df['locale_name'] == train_oil_store_holiday_df['city']),
    'holiday_flag'
] = 1


In [ ]:
train_oil_store_holiday_df.head(5)

In [ ]:
train_df_final = train_oil_store_holiday_df

In [ ]:
train_df_final.columns

In [ ]:
train_df_final.info()

In [ ]:
holidays_events_df.columns

In [ ]:
train_df_final.drop(["type_y", "type_x", "description", "transferred"], axis=1, inplace=True)

In [ ]:
train_df_final.info()

In [ ]:
train_df_full = train_df_final.merge(transactions_df, on=["store_nbr", "date"], how="left")

In [ ]:
train_df_full.isnull().sum()

In [ ]:
train_df_full.drop(["locale", "locale_name"], axis=1, inplace=True)

In [ ]:
train_df_full.info()

In [ ]:
train_df_full['holiday_flag'].fillna(0, inplace=True)
train_df_full['transactions'].fillna(0, inplace=True)
train_df_full.isnull().sum()

## 🔹 Part 2 — Feature Engineering

In [ ]:
import pandas as pd
import os

In [ ]:
# Define base folder path
DATA_PATH = "store_sales_data"

In [ ]:
# train_df_full.to_csv(os.path.join(DATA_PATH, "train_df_full.csv"), index=False)

In [ ]:
train_df_full = pd.read_csv(os.path.join(DATA_PATH, "train_df_full.csv"), parse_dates=["date"])

In [ ]:
train_df_full.columns

In [ ]:
train_df_full.info()

In [ ]:
train_df_full['year']=train_df_full['date'].dt.year
train_df_full['month']=train_df_full['date'].dt.month
train_df_full['day']=train_df_full['date'].dt.day
train_df_full['day_of_week']=train_df_full['date'].dt.dayofweek
train_df_full['week_of_year']=train_df_full['date'].dt.isocalendar().week
train_df_full['is_weekend'] = train_df_full['day_of_week'].isin([5, 6]).astype(int)

In [ ]:
train_df_full.info()

In [ ]:
train_sorted_df = train_df_full.sort_values(by=['store_nbr', 'family', 'date'])
lags = [1,7]
for lag in lags:
    train_sorted_df[f'lag_{lag}']=train_sorted_df.groupby(by=['store_nbr', 'family'])['sales'].shift(lag)

In [ ]:
grouped_sales = train_sorted_df.groupby(by=['store_nbr', 'family'])['sales']
train_sorted_df['rolling_mean_7'] = grouped_sales.transform( lambda x: x.shift(1).rolling(window=7).mean())
train_sorted_df['rolling_var_7'] = grouped_sales.transform( lambda x: x.shift(1).rolling(window=7).std())

In [ ]:
grouped_promo = train_sorted_df.groupby(by=['store_nbr', 'family'])['onpromotion']
train_sorted_df['promo_rolling_mean_7'] = grouped_promo.transform(lambda x: x.shift(1).rolling(window=7).mean())

In [ ]:
train_sorted_df.info()

In [ ]:
li = ['cluster','state', 'store_nbr', 'city', 'family']
for i in li:
    print(len(train_sorted_df[i].unique()))

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
encoder = {}
for l in li:
    le = LabelEncoder()
    train_sorted_df[f"{l}_encoded"] = le.fit_transform(train_sorted_df[l])
    encoder[l] = le
    train_sorted_df.drop(l, axis=1, inplace=True)

In [ ]:
import pickle
file_name = 'LabelEncoder.pkl'
with open(file_name, 'wb') as file:
    pickle.dump(encoder, file)

In [ ]:
train_sorted_df.to_csv(os.path.join(DATA_PATH, "df_final.csv"), index=False)

## 🔹 Part 3 — Model Training

In [2]:
import pandas as pd
import os
# Define base folder path
DATA_PATH = "store_sales_data"
train_df_final = pd.read_csv(os.path.join(DATA_PATH, "df_final.csv"), parse_dates=["date"])

In [3]:
train_df_final.drop('date',axis=1, inplace=True)

In [4]:
train_df_final.isnull().sum()

id                          0
sales                       0
onpromotion                 0
dcoilwtico                  0
holiday_flag                0
transactions                0
year                        0
month                       0
day                         0
day_of_week                 0
week_of_year                0
is_weekend                  0
lag_1                    1782
lag_7                   12474
rolling_mean_7          12474
rolling_var_7           12474
promo_rolling_mean_7    12474
cluster_encoded             0
state_encoded               0
store_nbr_encoded           0
city_encoded                0
family_encoded              0
dtype: int64

In [5]:
train_df_final=train_df_final.fillna(0)

In [6]:
train_df_final.isnull().sum()

id                      0
sales                   0
onpromotion             0
dcoilwtico              0
holiday_flag            0
transactions            0
year                    0
month                   0
day                     0
day_of_week             0
week_of_year            0
is_weekend              0
lag_1                   0
lag_7                   0
rolling_mean_7          0
rolling_var_7           0
promo_rolling_mean_7    0
cluster_encoded         0
state_encoded           0
store_nbr_encoded       0
city_encoded            0
family_encoded          0
dtype: int64

In [7]:
from sklearn.model_selection import train_test_split

X = train_df_final.drop('sales', axis=1)  # features
y = train_df_final['sales']               # target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
from sklearn.linear_model import LinearRegression

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Linear Regression model has been trained successfully! 🚀")
print(f"Intercept (the 'b' in y = mx + b): {model.intercept_}")
print(f"Coefficients (the 'm's' in y = m1x1 + m2x2 + ... + b): {model.coef_}")

In [ ]:
import pickle

filename = 'linear_regression_model.pkl'

# Open the file in write binary mode and save the model
with open(filename, 'wb') as file:
    pickle.dump(model, file)

print(f"Model saved to {filename} 📦")

## Part 4 Testing

In [1]:
import pickle
import numpy as np # To create sample data for prediction

# Open the file in read binary mode and load the model
with open('linear_regression_model.pkl', 'rb') as file:
    loaded_model = pickle.load(file)

print("Model loaded successfully from .pkl file! ✅")

Model loaded successfully from .pkl file! ✅


In [10]:
loaded_model.score(X_train, y_train)

0.9103177529752318

In [11]:
from sklearn.metrics import root_mean_squared_error

y_predicted = loaded_model.predict(X_test)
remse_ = root_mean_squared_error(y_test, y_predicted)
print(remse_)

316.9389705560476


In [12]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, y_predicted)
print(f"Mean Absolute Error (MAE): {mae}")

Mean Absolute Error (MAE): 89.14491121645047


In [13]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_predicted)
print(f"R-squared (R²): {r2}")

R-squared (R²): 0.9170981496268263
